In [23]:
# Data handling
import pandas as pd

# ML tools
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Model
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [24]:
# Load dataset (exported from preprocessing step)
df = pd.read_csv("../data/processed/processed_data.csv")

print(df.shape)
df.head()

(25000, 21)


,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_rating,delivery_cost,...,speed_kmph_recon,weather_mult_recon,delivery_time_hours_recon,partner_mult_recon,order_date_recon,order_ts_recon,expected_ts_recon,hour,delay_hours,delayed_flag
0,amazon logistics,automobile parts,ev bike,standard,west,clear,235.6,48.07,5,1322.21,...,30,1.0,9.332489,1.001906,21-10-2024,2024-10-21 13:00:00,2024-10-23 17:48:00,13,-43.467511,0
1,amazon logistics,clothing,bike,express,central,stormy,81.8,45.51,2,595.53,...,35,1.1,4.129935,1.001906,02-01-2024,2024-01-02 12:00:00,2024-01-02 20:00:00,12,-3.870065,0
2,amazon logistics,clothing,van,same day,north,clear,282.9,31.33,2,1608.49,...,45,1.0,7.427398,1.001906,31-05-2024,2024-05-31 11:00:00,2024-06-01 13:24:00,11,-18.972602,0
3,amazon logistics,cosmetics,ev bike,two day,central,hot,88.6,8.67,3,469.01,...,30,1.1,3.997011,1.001906,03-01-2024,2024-01-03 17:00:00,2024-01-05 17:00:00,17,-44.002989,0
4,amazon logistics,cosmetics,ev van,two day,east,rainy,204.2,8.09,4,1045.27,...,40,1.2,6.933351,1.001906,19-03-2024,2024-03-19 13:00:00,2024-03-21 17:48:00,13,-45.866649,0


In [25]:
# Target variable
y = df["delayed_flag"]

# Features
X = df.drop(columns=["delayed_flag", "delay_hours"])

In [26]:
# --------------------------------------------------
# Remove raw datetime columns (avoid leakage)
# --------------------------------------------------

X = X.drop(columns=[
    "order_date_recon",
    "order_ts_recon",
    "expected_ts_recon"
], errors="ignore")

In [27]:
# --------------------------------------------------
# Remove remaining leakage columns
# --------------------------------------------------

leakage_cols = [
    "delivery_time_hours_recon",   # actual delivery duration (future info)
]

X = X.drop(columns=leakage_cols, errors="ignore")

In [28]:
categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(exclude="object").columns

print(categorical_cols)
print(numerical_cols)

Index(['delivery_partner', 'package_type', 'vehicle_type', 'delivery_mode',
       'region', 'weather_condition'],
      dtype='object')
Index(['distance_km', 'package_weight_kg', 'delivery_rating', 'delivery_cost',
       'expected_time_hours_recon', 'speed_kmph_recon', 'weather_mult_recon',
       'partner_mult_recon', 'hour'],
      dtype='object')


In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)

In [31]:
model = RandomForestClassifier(random_state=42)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])

In [32]:
pipeline.fit(X_train, y_train)
print("Training completed")

Training completed


In [33]:
y_pred = pipeline.predict(X_test)

In [34]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9976

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      4853
           1       0.98      0.94      0.96       147

    accuracy                           1.00      5000
   macro avg       0.99      0.97      0.98      5000
weighted avg       1.00      1.00      1.00      5000


Confusion Matrix:
 [[4850    3]
 [   9  138]]


# SMOTE for Imbalance

In [35]:
!pip install imbalanced-learn


   ---------------------------------------- 0/2 [sklearn-compat]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from imblearn.over_sampling import SMOTE

In [38]:
# Check imbalance
print(y_train.value_counts()) 

delayed_flag
0    19412
1      588
Name: count, dtype: int64


In [39]:
# Transform training data using preprocessing pipeline
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [40]:
# Apply SMOTE only on training data

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_processed,
    y_train
)

In [41]:
print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())

Before SMOTE:
delayed_flag
0    19412
1      588
Name: count, dtype: int64

After SMOTE:
delayed_flag
0    19412
1    19412
Name: count, dtype: int64


In [42]:

# Train Random Forest on balanced data


model_smote = RandomForestClassifier(random_state=42)

model_smote.fit(X_train_smote, y_train_smote)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [43]:
# Predictions
y_pred_smote = model_smote.predict(X_test_processed)

In [44]:

# Evaluation after SMOTE


from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_smote))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote))

Accuracy: 0.9978

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4853
           1       0.96      0.97      0.96       147

    accuracy                           1.00      5000
   macro avg       0.98      0.98      0.98      5000
weighted avg       1.00      1.00      1.00      5000


Confusion Matrix:
[[4847    6]
 [   5  142]]


### After establishing a baseline Random Forest classifier, SMOTE was applied to address class imbalance. The SMOTE-enhanced model improved recall for delayed deliveries from 0.94 to 0.97, reducing missed delay predictions while maintaining high overall accuracy. This demonstrates improved minority-class detection without significant performance trade-offs.